<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB17_Choosing_an_Architecture_Closing_Deep_Learning_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB17 · Clase 17 — Elegir una arquitectura: cierre del Bloque 3**

## Bloque 3: IA — Deep Learning (cierre)

`NB11`–`NB16` introdujeron, cada uno, una arquitectura sobre un problema real. La pregunta que responde esta clase de cierre no es "qué es una CNN" — es **"dado un problema nuevo y nunca visto, ¿a cuál de estas herramientas recurres, y por qué?"** Construimos un marco de decisión explícito, lo aplicamos a todos los datasets reales usados hasta ahora en el curso, y después **demostramos** una de sus afirmaciones centrales con un experimento real: entrenar una red neuronal sobre un dataset lo bastante pequeño como para que el Machine Learning clásico (`NB10`) debería seguir ganando — `comprobando si de verdad ocurre así, en vez de limitarnos a afirmarlo`.

### Objetivos de aprendizaje

Al terminar esta clase, el alumnado será capaz de:
- Resumir para qué sirve cada arquitectura del Bloque 3, en una frase cada una.
- Aplicar un marco de decisión concreto (tipo de dato, disponibilidad de etiquetas, tamaño del dataset) a un problema nuevo.
- Explicar, con evidencia real, por qué el Deep Learning no gana automáticamente al ML clásico — y cuándo sí lo hace.
- Comparar una red neuronal con un modelo clásico ajustado de forma justa, sobre los mismos datos reales y el mismo split.
- Describir, a alto nivel, qué viene en el Bloque 4.

### Agenda (clase de 2 horas)

| # | Segmento de la clase | Duración aprox. | Tipo |
|---|---------------------|:---:|:---:|
| 1 | Repaso, hoja de ruta de hoy | 5 min | Teoría |
| 2 | Síntesis: todo el Bloque 3 en una tabla | 15 min | Teoría |
| 3 | Un marco de decisión para elegir arquitectura | 15 min | Teoría + Práctica |
| 4 | Aplicar el marco a todos los datasets usados en este bloque | 15 min | Teoría + Práctica |
| 5 | Práctica: ¿una red neuronal supera al Gradient Boosting ajustado de NB10? | 20 min | Práctica |
| 6 | Interpretar el resultado | 15 min | Práctica |
| 7 | Cuándo el Deep Learning *sí* gana: revisando la evidencia | 10 min | Teoría |
| 8 | Mirando hacia adelante: Bloque 4 | 10 min | Teoría |
| 9 | Una checklist para abordar un problema nuevo | 10 min | Teoría |
| 10 | Resumen, tarea | 5 min | Teoría |

> Los tiempos son orientación aproximada, no un guion cerrado — no hay descansos programados. Si se cubre todo con tiempo de sobra, la clase termina antes; puede pasar y no hay problema.

---

## 1. Repaso: dónde estamos

Seis clases, seis problemas reales, una pregunta de cierre: sabiendo todo esto, ¿cómo *decides* de verdad qué construir la próxima vez, antes de haberte pasado dos horas construyendo lo que no tocaba?

---

## 2. Síntesis: todo el Bloque 3 en una tabla

| Clase | Arquitectura | Idea central | Datos reales usados |
|---|---|---|---|
| `NB11` | Perceptrón / MLP | Sumas ponderadas + activaciones no lineales, apiladas en capas | Sonar (mina vs. roca) |
| `NB12` | Disciplina de entrenamiento | Monitorización con validación, dropout, parada temprana, elección de optimizador | Sonar |
| `NB13` | CNN | Filtros aprendidos y compartidos que respetan la estructura espacial | Imágenes submarinas LIACi |
| `NB14` | RNN / LSTM | Una celda reutilizada en cada paso temporal, que arrastra un estado oculto hacia delante | Secuencias mensuales de combustible del buque |
| `NB15` | Transfer learning | Reutilizar los filtros genéricos de una red preentrenada en vez de aprender desde cero | Imágenes LIACi + ResNet18 preentrenada |
| `NB16` | Autoencoder | Comprimir a través de un cuello de botella; el error de reconstrucción señala anomalías, sin supervisión | Datos de combustible del buque |

Fíjate en lo que *no* aparece en esta tabla: una única arquitectura "mejor". `Cada una respondió a un tipo concreto de pregunta, sobre un tipo concreto de dato`. Esa es la verdadera lección de este bloque, hecha explícita hoy.

---

## 3. Un marco de decisión para elegir arquitectura

Tres preguntas, hechas en orden, cubren casi todo lo visto en el curso hasta ahora:

1. **¿Tienes etiquetas?** Si no, estás en territorio de `NB09`/`NB16` (clustering, PCA, autoencoders) — ninguna elección de arquitectura importa hasta responder esto.
2. **¿Qué forma tiene el dato?** ¿Tabular/estructurado (`NB07`/`NB08`/`NB11`), espacial/imagen (`NB13`/`NB15`), o secuencial/dependiente del tiempo (`NB14`)? Esto determina la *familia* de arquitectura, en gran medida independientemente de cuántos datos tengas.
3. **¿Cuántos datos tienes?** Esto determina si entrenas desde cero, recurres a transfer learning, o prescindes por completo del Deep Learning a favor de un modelo clásico de `NB07`/`NB08`/`NB10` — exactamente la pregunta que esta clase pone a prueba experimentalmente en la Parte 5.

Dibujemos esto como un diagrama de flujo — con la forma deliberada de los árboles de decisión de `NB08`, porque estructuralmente es exactamente eso:

Dibuja el marco de decisión como un diagrama de flujo:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(12, 7))

def box(x, y, w, h, text, color="lightblue"):
    ax.add_patch(patches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05",
                                         facecolor=color, edgecolor="black"))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=8.5)

def arrow(x1, y1, x2, y2, label=""):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", lw=1.3))
    if label:
        ax.text((x1 + x2) / 2 + 0.15, (y1 + y2) / 2, label, fontsize=8, color="darkblue")

box(4, 9, 3, 1, "Do you have\nlabels?", "khaki")
arrow(4, 9.5, 1, 7.5, "No")
arrow(6.5, 9.5, 8.5, 7.5, "Yes")

box(-0.5, 6.5, 3, 1, "Clustering / PCA (NB09)\nor Autoencoder (NB16)", "lightgreen")

box(7, 6.5, 3, 1, "What shape\nis the data?", "khaki")
arrow(7.2, 6.5, 4.5, 4.5, "Tabular")
arrow(8.5, 6.5, 8.5, 4.5, "Image")
arrow(9.8, 6.5, 12.5, 4.5, "Sequence")

box(2.5, 3.5, 3, 1, "How much\ndata?", "khaki")
arrow(2.8, 3.5, 0.5, 1.5, "Small/medium")
arrow(4.3, 3.5, 5.5, 1.5, "Large")

box(-0.7, 1, 3, 1, "Classical ML\n(NB07/NB08/NB10)", "lightcoral")
box(4.3, 1, 3, 1, "MLP, but compare\nto classical ML (NB11)", "lightcoral")

box(7, 3.5, 3, 1, "Enough data to\ntrain from scratch?", "khaki")
arrow(7.3, 3.5, 6, 1.5, "No")
arrow(8.7, 3.5, 10, 1.5, "Yes")
box(4.7, 1, 3, 1.3, "Transfer learning\n(NB15)", "lightcoral")
box(8.7, 1, 3, 1.3, "CNN from scratch\n(NB13)", "lightcoral")

box(11, 3.5, 3, 1.3, "RNN / LSTM\n(NB14)", "lightcoral")

ax.set_xlim(-1.5, 15)
ax.set_ylim(0, 10.5)
ax.axis("off")
ax.set_title("A decision framework for choosing an architecture")
plt.tight_layout()
plt.show()

Esta es una guía simplificada, no un libro de reglas rígido — los problemas reales a veces caen entre ramas (datos tabulares con millones de filas podrían de verdad soportar un MLP; un dataset de imágenes pequeño podría funcionar mejor con augmentación de datos intensiva en vez de transfer learning). `Trátalo como un prior fuerte desde el que empezar a razonar, no como una tabla de consulta`.

> **Para saber más**: [Teorema de no free lunch (Wikipedia)](https://en.wikipedia.org/wiki/No_free_lunch_theorem) — la razón formal por la que ningún algoritmo domina en todos los problemas posibles, y por qué un marco como este tiene que seguir siendo una heurística, no una ley.

**Pruébalo tú mismo**: convierte el diagrama de flujo anterior en una función real e invocable — introduce la forma de un problema hipotético nuevo y mira qué recomienda.

In [ ]:
def suggest_architecture(has_labels, data_shape, n_examples):
    if not has_labels:
        return "Clustering/PCA (NB09) or Autoencoder (NB16)"
    if data_shape == "sequence":
        return "RNN/LSTM (NB14)"
    if data_shape == "image":
        return "CNN from scratch (NB13)" if n_examples >= 5000 else "Transfer learning (NB15)"
    # tabular
    return "Classical ML (NB07/NB08/NB10)" if n_examples < 5000 else "MLP, but compare to classical ML (NB11)"

print(suggest_architecture(has_labels=True, data_shape="image", n_examples=50000))
print(suggest_architecture(has_labels=True, data_shape="tabular", n_examples=100))


---

## 4. Aplicar el marco a todos los datasets usados en este bloque

Recorre cada dataset real de `NB11`–`NB16` a través de las tres preguntas:

| Dataset | ¿Etiquetas? | Forma | Volumen de datos | El marco dice | Qué hicimos en realidad |
|---|---|---|---|---|---|
| Sonar (mina/roca) | Sí | Tabular (60 características) | 208 filas — poco | ML clásico, o un MLP modesto | `NB08`: ensembles clásicos/SVM. `NB11`/`NB12`: MLP, competitivo pero no claramente mejor |
| Imágenes LIACi | Sí | Imagen | 1.893 imágenes — modesto para imágenes | Transfer learning probablemente gana al entrenamiento desde cero | `NB13`: desde cero. `NB15`: transfer learning — la lógica de la Parte 5 predice que esto debería ayudar en general |
| Secuencias de combustible del buque | Sí | Secuencial | 120 secuencias — poco | RNN/LSTM arquitectónicamente correcto, pero necesita muchos datos | `NB14`: LSTM frente a una base ingenua — una comparación justa dado lo pocos datos que había |
| Datos de combustible del buque (anomalías) | No | Tabular | 1.440 filas | Sin supervisión: clustering/PCA o autoencoder | `NB09`: K-Means/PCA. `NB16`: autoencoder |

El patrón que merece la pena notar: **la predicción del marco y lo que la evaluación honesta de cada clase encontró de verdad coinciden** — los datos tabulares pequeños hicieron competitivo al ML clásico en `NB08`/`NB11`, el transfer learning tuvo una ventaja teórica real sobre la CNN desde cero en `NB13`/`NB15`, y la LSTM de `NB14` nunca prometió una victoria fácil porque no había datos suficientes para ello. `Esto no es una coincidencia — es el marco haciendo su trabajo`.

**Pruébalo tú mismo**: pasa los datasets reales de la tabla de este bloque por `suggest_architecture` — ¿coincide la salida de la función con lo que ya afirma la columna "El marco dice" de arriba?

In [ ]:
datasets = [
    ("Sonar (mine/rock)", True, "tabular", 208),
    ("LIACi images", True, "image", 1893),
    ("Ship fuel sequences", True, "sequence", 120),
    ("Ship fuel data (anomalies)", False, "tabular", 1440),
]

for name, has_labels, shape, n in datasets:
    print(f"{name}: {suggest_architecture(has_labels, shape, n)}")


---

## 5. Práctica: ¿una red neuronal supera al Gradient Boosting ajustado de `NB10`?

Es hora de dejar de afirmar que "los datos tabulares pequeños favorecen al ML clásico" y ponerlo a prueba de verdad. `NB10` construyó un proyecto completo y ajustado sobre el dataset real **Yacht Hydrodynamics** (308 medidas reales de casco) e informó del rendimiento en test de un modelo de Gradient Boosting ajustado con `GridSearchCV`. Entrenemos un MLP pequeño sobre **exactamente los mismos datos y el mismo split**, y comparemos con honestidad.

In [ ]:
!wget -q -O yacht.data https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/yacht_hydrodynamics.data

import pandas as pd

columns = [
    "LongPos_COB", "Prismatic_Coeff", "LengthDisp_Ratio",
    "BeamDraft_Ratio", "LengthBeam_Ratio", "Froude_Number", "Residuary_Resistance",
]
yacht = pd.read_csv("yacht.data", sep=r"\s+", names=columns)

X = yacht.drop(columns="Residuary_Resistance")
y = yacht["Residuary_Resistance"]
print(yacht.shape)

Mismo split que `NB10` (`test_size=0.2, random_state=42`), para que la comparación sea justa — sin ventaja para ningún lado por un split afortunado:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

Un MLP pequeño — deliberadamente parecido en espíritu al `SonarMLP` de `NB11`, con dropout siguiendo el manual de `NB12` para mantenerlo honesto con tan pocos datos:

In [ ]:
import torch.nn as nn

class YachtMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_features, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.layers(x)

torch.manual_seed(42)
yacht_mlp = YachtMLP(n_features=X_train_t.shape[1])

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(yacht_mlp.parameters(), lr=0.01)

n_epochs = 300
for epoch in range(n_epochs):
    yacht_mlp.train()
    optimizer.zero_grad()
    loss = criterion(yacht_mlp(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

print(f"Final training loss: {loss.item():.3f}")

Evalúa con las mismas métricas exactas de `NB10`:

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

yacht_mlp.eval()
with torch.no_grad():
    mlp_pred = yacht_mlp(X_test_t).numpy().ravel()

mlp_mae = mean_absolute_error(y_test, mlp_pred)
mlp_rmse = mean_squared_error(y_test, mlp_pred) ** 0.5
mlp_r2 = r2_score(y_test, mlp_pred)

print(f"MLP  - MAE: {mlp_mae:.3f}  RMSE: {mlp_rmse:.3f}  R2: {mlp_r2:.3f}")
print("Compare against NB10's tuned Gradient Boosting test-set numbers (re-run NB10 Part 8 if you don't have them handy).")

**Pruébalo tú mismo**: no te limites a volver a ejecutar `NB10` por separado — replica aquí su modelo exacto de Gradient Boosting ajustado, sobre este mismo split, para que la comparación quede totalmente autocontenida en un único notebook.

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor

cv = KFold(n_splits=5, shuffle=True, random_state=42)
tuning_pipe = Pipeline([("scaler", StandardScaler()), ("model", GradientBoostingRegressor(random_state=42))])
param_grid = {
    "model__n_estimators": [100, 300, 500],
    "model__max_depth": [3, 5, 10],   # 3 is GradientBoostingRegressor's own default
    "model__min_samples_leaf": [1, 2, 4],
}
grid = GridSearchCV(tuning_pipe, param_grid, cv=cv, scoring="r2", n_jobs=-1)
grid.fit(X_train, y_train)

gb_pred = grid.best_estimator_.predict(X_test)
gb_mae = mean_absolute_error(y_test, gb_pred)
gb_rmse = mean_squared_error(y_test, gb_pred) ** 0.5
gb_r2 = r2_score(y_test, gb_pred)

print(f"Gradient Boosting (tuned here) - MAE: {gb_mae:.3f}  RMSE: {gb_rmse:.3f}  R2: {gb_r2:.3f}")
print(f"MLP (above)                     - MAE: {mlp_mae:.3f}  RMSE: {mlp_rmse:.3f}  R2: {mlp_r2:.3f}")


Un diagrama de barras hace que la comparación de las tres métricas se lea de un vistazo:

In [ ]:
comparison = pd.DataFrame([
    {"Model": "MLP", "MAE": mlp_mae, "RMSE": mlp_rmse, "R2": mlp_r2},
    {"Model": "Gradient Boosting (tuned)", "MAE": gb_mae, "RMSE": gb_rmse, "R2": gb_r2},
])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, metric in zip(axes, ["MAE", "RMSE", "R2"]):
    ax.bar(comparison["Model"], comparison[metric], color=["steelblue", "darkorange"])
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


---

## 6. Interpretar el resultado

**Lee tus propios números, con honestidad** — esta sección tiene tres desenlaces honestos posibles, y solo uno de ellos es el "esperado":

- **Gana Gradient Boosting**: el resultado esperado, y buena evidencia para el marco de la Parte 3 — 308 filas son de verdad pocos datos para que una red con cientos de pesos entrenables aprenda de ellos, mientras que la búsqueda con `GridSearchCV` de `NB10` podía explotar la necesidad de datos mucho menor del ensemble de árboles.
- **Quedan cerca**: también es un desenlace legítimo y habitual — 6 características numéricas limpias y bien comportadas con una relación bastante suave con el objetivo es un problema de regresión relativamente *fácil* para ambos enfoques.
- **Gana el MLP**: menos esperado, pero no imposible — merece una segunda mirada comprobar si el modelo clásico estaba infra-ajustado, o si el split de esta semilla aleatoria en particular favoreció a la red por casualidad.

Sea cual sea el resultado, tienes **evidencia real**, no una afirmación repetida — que es todo el sentido de ejecutar esta comparación en vez de limitarse a afirmarla en una viñeta.

---

## 7. Cuándo el Deep Learning *sí* gana: revisando la evidencia

La Parte 5 puso a prueba un lado del marco. El otro lado también tiene evidencia en este bloque:

- `NB13`/`NB15`: la CNN desde cero tuvo que aprender cada filtro — bordes incluidos — a partir de ~400 imágenes. El transfer learning partió de filtros ya entrenados sobre 1,4 millones de imágenes, necesitando muchas menos épocas de entrenamiento para llegar a un resultado comparable o mejor en la misma tarea real.
- La tabla motivadora original de `NB01` §3: las victorias más claras del Deep Learning siguen estando en **datos crudos y no estructurados** (imágenes, audio, secuencias largas, texto) — precisamente el tipo de dato que el ML clásico del Bloque 2 nunca estuvo pensado para manejar directamente.

El retrato honesto y completo de todo este bloque: el Deep Learning no es "mejor" que el ML clásico en general — `es un conjunto de herramientas distinto, con una ventaja real concentrada en situaciones específicas` (datos no estructurados, datasets grandes, disponibilidad de transfer learning) que el marco de la Parte 3 intenta nombrar explícitamente, en vez de dejarlo como una intuición vaga.

---

## 8. Mirando hacia delante: Bloque 4

**Bloque 4 — Proyectos: casos de estudio** aplica todo lo visto en los Bloques 2 y 3 a casos de estudio reales y completos, usando datasets navales/oceánicos que todavía no se han usado como ejemplo principal en este curso — datos meteorológicos, registros históricos de travesías, y datos de terreno/elevación. Cada uno supondrá elegir tu propia arquitectura usando exactamente el razonamiento construido hoy, sin que se te diga cuál usar.

---

## 9. Una checklist para abordar un problema nuevo

Antes de escribir una sola línea de código de modelo sobre un problema nuevo:

1. **Enmárcalo.** ¿Qué es exactamente lo que predices o descubres, y por qué importa? (`NB10` Parte 2)
2. **Comprueba si hay etiquetas.** Supervisado o no supervisado — esto cambia todo lo que viene después. (Parte 3, pregunta 1)
3. **Conoce la forma de tus datos.** Tabular, imagen, o secuencial. (Parte 3, pregunta 2)
4. **Conoce el volumen de tus datos.** Basta con un orden de magnitud aproximado: decenas, cientos, miles, millones. (Parte 3, pregunta 3)
5. **Empieza simple, y después justifica la complejidad.** Una línea base clásica (`NB07`/`NB08`) se construye rápido y te dice si un modelo más sofisticado siquiera merece el esfuerzo — exactamente lo que la Parte 5 acaba de demostrar directamente.
6. **Evalúa con honestidad.** Datos de test reservados, tocados una sola vez (desde `NB07` en adelante); compara contra una línea base ingenua (`NB14`) o un resultado ya existente (como hoy), nunca solo contra tu propia pérdida de entrenamiento.
7. **Interpreta, no te limites a reportar.** Una matriz de confusión, un gráfico de importancia de características, una tabla cruzada de error de reconstrucción — cada clase real de este curso acompañó un número con una explicación de lo que significa operativamente.

---

## Resumen de la clase

- El Bloque 3 cubrió seis problemas reales, cada uno resuelto con la arquitectura que de verdad encajaba con sus datos — no la misma herramienta aplicada seis veces.
- Tres preguntas (¿etiquetas? ¿forma del dato? ¿volumen de datos?) cubren la mayor parte de la decisión de elegir arquitectura, visualizadas como un diagrama de flujo con la forma de los árboles de decisión de `NB08`.
- Pusimos a prueba directamente la afirmación de que "los datos tabulares pequeños favorecen al ML clásico", entrenando un MLP sobre el dataset y el split exactos de `NB10`, en vez de limitarnos a afirmarlo.
- Las ventajas reales del Deep Learning se concentran en datos no estructurados, datasets grandes, y situaciones donde hay transfer learning disponible — no en todas partes ni todo el tiempo.
- El Bloque 4 aplica todo lo visto en los Bloques 2–3 a nuevos casos de estudio reales, dejando la elección de arquitectura en tus manos.

## Tarea / Ideas de práctica

1. Vuelve a ejecutar la Parte 5 con `n_epochs` a 1000 en vez de 300 — ¿el MLP cierra algo de distancia con Gradient Boosting, o empieza a sobreajustar en este dataset tan pequeño en su lugar (compruébalo añadiendo un split de validación, al estilo de `NB12`)?
2. Elige un dataset del Bloque 2 (`Naval_Dataset.csv` de `NB02`, o los datos Sonar de `NB08`) y recórrelo desde cero por el diagrama de flujo de la Parte 3, anotando tu razonamiento en cada rama antes de comprobarlo contra lo que el curso hizo realmente.
3. Explica con tus propias palabras por qué la comparación de la Parte 5 usó *exactamente* el mismo split `random_state=42` que `NB10` — ¿qué tendría de malo comparar en su lugar contra un split aleatorio distinto y nuevo?
4. Usando la checklist de la Parte 9, escribe (en una celda markdown, sin necesidad de código) cómo abordarías uno de los próximos temas del Bloque 4 (previsión meteorológica, análisis histórico de travesías, o datos de terreno) antes de que esa clase te dé nunca una respuesta.

> ***Como siempre: un marco se gana su sitio produciendo predicciones comprobables — la Parte 5 es lo que convierte el argumento de esta clase en evidencia, no en opinión.***